# 🎯 PatchTST với Optuna - Tự Động Tìm Seed Tốt Nhất

Notebook này cải tiến từ `patchtst_best_method.ipynb` với tính năng **tự động tìm seed tốt nhất** cho Optuna:

## Tính năng mới:
1. **Optuna với seed**: Sử dụng `TPESampler(seed=SEED)` để đảm bảo reproducibility
2. **Tự động tìm seed tốt nhất**: Test nhiều seed khác nhau và chọn seed cho kết quả tốt nhất
3. **Lưu/load best parameters**: Lưu kết quả vào file JSON để tái sử dụng

## Phương pháp:
1. **Seed Optimization**: Test nhiều seed (1, 42, 100, 200, 300, ...) với Optuna (ít trials)
2. **Chọn seed tốt nhất**: Seed cho MSE thấp nhất
3. **Optuna đầy đủ**: Chạy Optuna với seed tốt nhất (20 trials đầy đủ)
4. **Post-processing**: Smooth Linear 20% + Post-processing Regression


## 1. Setup và Import Libraries


In [ ]:
# Cài đặt các thư viện cần thiết
import subprocess
import sys

def install_package(package, import_name=None):
    """Cài đặt package nếu chưa có"""
    if import_name is None:
        import_name = package
    try:
        __import__(import_name)
        print(f"✓ {package} đã được cài đặt")
        return True
    except ImportError:
        print(f"📦 Đang cài đặt {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package],
                                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"✓ Đã cài đặt {package}")
            return True
        except Exception as e:
            print(f"⚠️  Lỗi khi cài đặt {package}: {e}")
            return False

# Cài đặt các thư viện cần thiết
packages_to_install = [
    ('neuralforecast', 'neuralforecast'),
    ('optuna', 'optuna'),
    ('scikit-learn', 'sklearn'),
    ('scipy', 'scipy')
]

print("🔧 Kiểm tra và cài đặt các thư viện cần thiết...\n")
for package, import_name in packages_to_install:
    install_package(package, import_name)

print("\n✓ Hoàn thành kiểm tra/cài đặt thư viện!")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import json
import random
from pathlib import Path

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit

# Set encoding để tránh lỗi Unicode
import sys
import io
if sys.platform == 'win32':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8')

# NeuralForecast
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
import optuna
from optuna import Trial
from optuna.samplers import TPESampler

print("✓ Đã import các thư viện cần thiết")


## 2. Load và Chuẩn Bị Dữ Liệu


In [ ]:
# Load dữ liệu training
csv_path = Path("./FPT_train.csv")
if not csv_path.exists():
    # Thử download từ Google Drive
    from pathlib import Path
    import subprocess
    DRIVE_FILE_ID = "1nS9xshut38SJEX__PD_zjKFtj2CQCn7S"
    try:
        import gdown
        gdown.download(f"https://drive.google.com/uc?id={DRIVE_FILE_ID}", str(csv_path), quiet=False)
    except:
        print("⚠️  Vui lòng đảm bảo file FPT_train.csv tồn tại")
        raise

df = pd.read_csv(csv_path, parse_dates=["time"])
df = df.sort_values("time").reset_index(drop=True)

print(f"✓ Đã load dữ liệu training: {len(df)} điểm")
print(f"   - Từ {df['time'].min()} đến {df['time'].max()}")

# Chuẩn bị dữ liệu
target_col = "close"
horizon = 100  # Dự đoán 100 ngày tiếp theo

close_values = df[target_col].values.astype("float32")
T = len(close_values)

# Chia train/validation
train_size = int(T * 0.8)
val_size = int(T * 0.1)

train_data = close_values[:train_size]
val_data = close_values[train_size:train_size + val_size]

print(f"\n📊 Chia dữ liệu training:")
print(f"   - Train: {len(train_data)} điểm")
print(f"   - Val: {len(val_data)} điểm")


In [ ]:
# Download và load file test từ Google Drive
from pathlib import Path
import subprocess

# Google Drive file ID cho file test/ground truth
TEST_FILE_ID = "1IkzoSTHPMnOUBILN7cCPjVw9QWAuOtCs"
test_file_path = Path("./FPT_test.csv")

# Download file test
if test_file_path.exists():
    print(f"✓ File test đã tồn tại tại: {test_file_path}")
else:
    print("📥 Đang download file test từ Google Drive...")

    try:
        try:
            import gdown
        except ImportError:
            print("   Đang cài đặt gdown...")
            subprocess.run(["pip", "install", "-q", "gdown"], check=True)
            import gdown

        gdown.download(f"https://drive.google.com/uc?id={TEST_FILE_ID}", str(test_file_path), quiet=False)

        if test_file_path.exists():
            print(f"✓ Đã download file test thành công tại: {test_file_path}")
        else:
            raise Exception("Download không thành công")
    except Exception as e:
        print(f"❌ Lỗi khi download: {e}")
        raise

# Đọc và lọc file test theo symbol FPT và đúng ngày
df_test_raw = pd.read_csv(test_file_path, parse_dates=["time"] if "time" in pd.read_csv(test_file_path, nrows=1).columns else None)

print(f"\n📊 Cấu trúc file test (trước khi lọc):")
print(f"   - Tổng số dòng: {len(df_test_raw):,}")

# Lọc theo symbol FPT
if "symbol" in df_test_raw.columns:
    df_test = df_test_raw[df_test_raw["symbol"] == "FPT"].copy()
    print(f"   - Sau khi lọc theo symbol='FPT': {len(df_test):,} dòng")

    # Sắp xếp theo thời gian
    if "time" in df_test.columns:
        df_test = df_test.sort_values("time").reset_index(drop=True)

        # Lấy ngày cuối cùng từ training data để lọc đúng ngày
        last_train_date = df["time"].max()
        print(f"   - Ngày cuối cùng trong training: {last_train_date.strftime('%Y-%m-%d')}")

        # Lọc các ngày sau ngày cuối cùng của training
        df_test = df_test[df_test["time"] > last_train_date].copy()
        df_test = df_test.sort_values("time").reset_index(drop=True)
        print(f"   - Sau khi lọc ngày > {last_train_date.strftime('%Y-%m-%d')}: {len(df_test):,} dòng")
else:
    print("⚠️  File không có cột 'symbol'. Sử dụng toàn bộ file:")
    df_test = df_test_raw.copy()
    if "time" in df_test.columns:
        df_test = df_test.sort_values("time").reset_index(drop=True)

# Chuẩn bị dữ liệu cho NeuralForecast
train_nf = pd.DataFrame({
    "unique_id": "FPT",
    "ds": df["time"].iloc[:train_size],
    "y": train_data
})

val_nf = pd.DataFrame({
    "unique_id": "FPT",
    "ds": df["time"].iloc[train_size:train_size + val_size],
    "y": val_data
})

# Lấy ground truth từ test data (100 điểm đầu)
test_ground_truth = df_test[target_col].iloc[:horizon].values.astype("float32")

# Tạo full training data cho Optuna
train_nf_full = pd.DataFrame({
    "unique_id": "FPT",
    "ds": df["time"].iloc[:train_size + val_size],
    "y": close_values[:train_size + val_size]
})

print(f"\n✓ Đã lấy {len(test_ground_truth)} điểm ground truth từ test data để đánh giá")
print(f"\n✓ Đã chuẩn bị dữ liệu cho NeuralForecast")
print(f"   - Train: {len(train_nf)} điểm")
print(f"   - Val: {len(val_nf)} điểm")
print(f"   - Test ground truth: {len(test_ground_truth)} điểm")


## 3. Tối Ưu Seed Tốt Nhất cho Optuna

Test nhiều seed khác nhau để tìm seed cho kết quả tốt nhất:


In [ ]:
print("="*70)
print("🔧 TỐI ƯU HYPERPARAMETERS VỚI OPTUNA CHO PATCHTST")
print("="*70)

# Chia dữ liệu cho Optuna optimization
# Dùng 90% đầu để train, 10% cuối để validation
optuna_train_size = int(len(train_nf_full) * 0.9)
train_nf_optuna = train_nf_full.iloc[:optuna_train_size].copy()
val_nf_optuna = train_nf_full.iloc[optuna_train_size:].copy()

# Lấy validation ground truth để đánh giá trong Optuna
val_close_optuna = val_nf_optuna["y"].values.astype("float32")

print(f"\n📊 Dữ liệu cho optimization:")
print(f"   - Train cho Optuna: {len(train_nf_optuna)} điểm (90%)")
print(f"   - Val cho Optuna: {len(val_nf_optuna)} điểm (10%)")
print(f"   - Horizon: {horizon}")
print(f"\n💡 Optuna sẽ tối ưu hyperparameters cho PatchTST")
print(f"   Sau khi tìm được best params, sẽ train final model trên TOÀN BỘ {len(train_nf_full)} điểm.")


In [ ]:
# Objective function cho Optuna
def objective_patchtst(trial: Trial, seed: int = None):
    """Objective function cho PatchTST optimization"""
    # Set seed nếu được cung cấp
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        try:
            import torch
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
        except:
            pass
    
    # Suggest hyperparameters
    input_size = trial.suggest_int("input_size", 100, 300, step=50)
    patch_len = trial.suggest_int("patch_len", 8, 32, step=8)
    stride = trial.suggest_int("stride", 4, 16, step=4)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    max_steps = trial.suggest_int("max_steps", 50, 300, step=50)
    
    try:
        model = PatchTST(
            h=min(horizon, len(val_nf_optuna)),
            input_size=input_size,
            patch_len=patch_len,
            stride=stride,
            revin=True,
            learning_rate=learning_rate,
            max_steps=max_steps,
            val_check_steps=10,
        )
        
        nf_model = NeuralForecast(models=[model], freq='D')
        nf_model.fit(df=train_nf_optuna, val_size=0)
        forecast = nf_model.predict()
        
        # Lấy predictions
        pred_col = [col for col in forecast.columns if col not in ['unique_id', 'ds']][0]
        pred = forecast[pred_col].values
        
        # Chỉ lấy số điểm tương ứng với validation
        n_points = min(len(pred), len(val_close_optuna), horizon)
        pred = pred[:n_points]
        val_true = val_close_optuna[:n_points]
        
        # Tính MSE
        mse = mean_squared_error(val_true, pred)
        return mse
    except Exception as e:
        print(f"   ⚠️  Lỗi trong trial: {e}")
        return float('inf')

print("✓ Đã định nghĩa objective function")


### 3.1. Test nhiều seed để tìm seed tốt nhất

**Cải tiến so với phiên bản cơ bản:**
1. **Nhiều seeds hơn**: Random search hoặc grid search seeds (20 seeds)
2. **Chỉ tính MSE**: Đơn giản, tập trung vào metric chính
3. **Statistical significance**: Chạy nhiều lần với cùng seed để kiểm tra độ ổn định
4. **Early stopping**: Dừng sớm nếu seed quá tệ (MSE > threshold)
5. **Stability check**: Tính độ lệch chuẩn MSE qua các lần chạy


In [ ]:
# ============================================================================
# Tìm seed tốt nhất dựa trên MSE
# ============================================================================

# Cấu hình
USE_RANDOM_SEED_SEARCH = True  # True: random search, False: grid search với danh sách cố định
N_SEEDS_TO_TEST = 20  # Số lượng seeds để test
N_TRIALS_PER_SEED = 8  # Số trials để đánh giá mỗi seed
N_RUNS_PER_SEED = 2  # Chạy mỗi seed nhiều lần để kiểm tra độ ổn định

# Early stopping: Dừng nếu seed quá tệ
USE_EARLY_STOPPING = True
EARLY_STOPPING_THRESHOLD = 500.0  # Nếu MSE > threshold, skip seed này

print("="*70)
print("🔍 TÌM SEED TỐT NHẤT CHO OPTUNA (DỰA TRÊN MSE)")
print("="*70)
print(f"📋 Cấu hình:")
print(f"   - Số seeds test: {N_SEEDS_TO_TEST}")
print(f"   - Mỗi seed: {N_TRIALS_PER_SEED} trials")
print(f"   - Mỗi seed chạy: {N_RUNS_PER_SEED} lần (kiểm tra độ ổn định)")
print(f"   - Early stopping: {'Có' if USE_EARLY_STOPPING else 'Không'}")
if USE_EARLY_STOPPING:
    print(f"   - Threshold: MSE > {EARLY_STOPPING_THRESHOLD}")
print()

# Tạo danh sách seeds để test
if USE_RANDOM_SEED_SEARCH:
    # Random search: chọn ngẫu nhiên từ range lớn
    base_seeds = [1, 42, 100, 200, 300, 500, 1000, 2024]  # Một số seeds "tốt" đã biết
    random_seeds = random.sample(range(1, 10000), N_SEEDS_TO_TEST - len(base_seeds))
    SEEDS_TO_TEST = base_seeds + random_seeds
    print(f"📌 Random search: {len(base_seeds)} seeds cố định + {len(random_seeds)} seeds ngẫu nhiên")
else:
    # Grid search: danh sách cố định
    SEEDS_TO_TEST = [1, 42, 100, 200, 300, 500, 1000, 2024, 5000, 7777, 8888, 9999]
    SEEDS_TO_TEST = SEEDS_TO_TEST[:N_SEEDS_TO_TEST]
    print(f"📌 Grid search: {len(SEEDS_TO_TEST)} seeds cố định")

print(f"   Seeds: {SEEDS_TO_TEST[:10]}..." if len(SEEDS_TO_TEST) > 10 else f"   Seeds: {SEEDS_TO_TEST}")
print()

# Test seeds - chỉ tính MSE
seed_results = []

for seed_idx, test_seed in enumerate(SEEDS_TO_TEST):
    print(f"\n{'='*70}")
    print(f"🌱 Test Seed {test_seed} ({seed_idx + 1}/{len(SEEDS_TO_TEST)})")
    print(f"{'='*70}")
    
    seed_mse_values = []
    
    # Chạy nhiều lần với cùng seed để kiểm tra độ ổn định
    for run in range(N_RUNS_PER_SEED):
        if run > 0:
            print(f"   🔄 Run {run + 1}/{N_RUNS_PER_SEED}...")
        
        # Reset seed
        random.seed(test_seed)
        np.random.seed(test_seed)
        try:
            import torch
            torch.manual_seed(test_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(test_seed)
                torch.cuda.manual_seed_all(test_seed)
        except:
            pass
        
        # Tạo study với seed
        sampler = TPESampler(seed=test_seed)
        study = optuna.create_study(
            direction='minimize',
            study_name=f'PatchTST_Seed{test_seed}_Run{run+1}',
            sampler=sampler
        )
        
        # Wrapper function với seed
        def objective_wrapper(trial):
            return objective_patchtst(trial, seed=test_seed)
        
        # Optimize
        study.optimize(objective_wrapper, n_trials=N_TRIALS_PER_SEED, show_progress_bar=False)
        
        # Early stopping: nếu MSE quá tệ, skip seed này
        if USE_EARLY_STOPPING and study.best_value > EARLY_STOPPING_THRESHOLD:
            print(f"   ⚠️  Seed {test_seed} quá tệ (MSE = {study.best_value:.4f} > {EARLY_STOPPING_THRESHOLD}), skip...")
            break
        
        # Lưu MSE
        seed_mse_values.append(study.best_value)
    
    # Tính trung bình và độ lệch chuẩn MSE qua các runs
    if len(seed_mse_values) > 0:
        avg_mse = np.mean(seed_mse_values)
        std_mse = np.std(seed_mse_values)
        
        seed_results.append({
            'seed': test_seed,
            'mse': avg_mse,
            'mse_std': std_mse,
            'best_params': study.best_params if 'study' in locals() else None
        })
        
        print(f"   ✅ Seed {test_seed}: MSE = {avg_mse:.4f} (±{std_mse:.4f})")

# Tìm seed tốt nhất dựa trên MSE
if len(seed_results) > 0:
    best_seed_result = min(seed_results, key=lambda x: x['mse'])
    BEST_SEED = best_seed_result['seed']
    
    print(f"\n{'='*70}")
    print("📊 KẾT QUẢ TEST SEEDS (Sắp xếp theo MSE):")
    print(f"{'='*70}")
    print(f"{'Seed':<8} {'MSE':<15} {'Stability (std)':<15}")
    print("-" * 40)
    
    for result in sorted(seed_results, key=lambda x: x['mse']):
        marker = "🏆" if result['seed'] == BEST_SEED else "  "
        print(f"{marker} {result['seed']:<6} {result['mse']:>13.4f} {result['mse_std']:>13.4f}")
    
    print(f"\n✅ SEED TỐT NHẤT: {BEST_SEED}")
    print(f"   - MSE: {best_seed_result['mse']:.4f}")
    print(f"   - Stability (MSE std): {best_seed_result['mse_std']:.4f}")
    print(f"   Sẽ sử dụng seed này cho Optuna đầy đủ")
else:
    print("\n⚠️  Không tìm được seed nào hợp lệ!")
    BEST_SEED = 42  # Fallback


### 3.2. Chạy Optuna đầy đủ với seed tốt nhất


In [ ]:
# Kiểm tra xem đã có best parameters chưa
BEST_PARAMS_FILE = Path("best_params_patchtst_best_seed.json")
USE_SAVED_PARAMS = False  # Set True nếu muốn load từ file

if USE_SAVED_PARAMS and BEST_PARAMS_FILE.exists():
    print("📂 Đã tìm thấy file best parameters. Loading...")
    with open(BEST_PARAMS_FILE, 'r', encoding='utf-8') as f:
        saved_data = json.load(f)
    best_params_patchtst = saved_data['best_params']
    best_mse_saved = saved_data['best_mse']
    BEST_SEED = saved_data.get('seed', BEST_SEED)
    print(f"\n✅ Sử dụng best parameters đã lưu:")
    print(f"   - Seed: {BEST_SEED}")
    for key, value in best_params_patchtst.items():
        print(f"   - {key}: {value}")
    print(f"   - Best MSE: {best_mse_saved:.4f}")
else:
    print(f"\n{'='*70}")
    print(f"🚀 CHẠY OPTUNA ĐẦY ĐỦ VỚI SEED TỐT NHẤT: {BEST_SEED}")
    print(f"{'='*70}")
    
    # Reset seed với best seed
    random.seed(BEST_SEED)
    np.random.seed(BEST_SEED)
    try:
        import torch
        torch.manual_seed(BEST_SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(BEST_SEED)
            torch.cuda.manual_seed_all(BEST_SEED)
    except:
        pass
    
    # Tạo study với best seed
    sampler = TPESampler(seed=BEST_SEED)
    study_patchtst = optuna.create_study(
        direction='minimize',
        study_name=f'PatchTST_BestSeed{BEST_SEED}',
        sampler=sampler
    )
    
    # Wrapper function với best seed
    def objective_with_best_seed(trial):
        return objective_patchtst(trial, seed=BEST_SEED)
    
    print(f"\n🔄 Đang tối ưu PatchTST với Optuna (20 trials)...")
    print(f"   - Seed: {BEST_SEED}")
    print(f"   ⏳ Quá trình này có thể mất vài phút...\n")
    
    study_patchtst.optimize(objective_with_best_seed, n_trials=20, show_progress_bar=True)
    
    # Lưu best parameters
    best_params_patchtst = study_patchtst.best_params
    best_mse_optuna = study_patchtst.best_value
    
    # Lưu vào file
    save_data = {
        'best_params': best_params_patchtst,
        'best_mse': best_mse_optuna,
        'seed': BEST_SEED,
        'timestamp': pd.Timestamp.now().isoformat()
    }
    with open(BEST_PARAMS_FILE, 'w', encoding='utf-8') as f:
        json.dump(save_data, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Best parameters cho PatchTST:")
    print(f"   - Seed: {BEST_SEED}")
    for key, value in best_params_patchtst.items():
        print(f"   - {key}: {value}")
    print(f"   - Best MSE: {best_mse_optuna:.4f}")
    print(f"\n💾 Đã lưu vào {BEST_PARAMS_FILE}")


## 4. Train PatchTST Baseline Model với Best Parameters

Sử dụng best parameters từ Optuna (với seed tốt nhất) để train model cuối cùng:


In [ ]:
# Train PatchTST với best parameters từ Optuna trên TOÀN BỘ dữ liệu
print("="*70)
print("🏋️  TRAIN PATCHTST VỚI BEST PARAMETERS")
print("="*70)

# Set seed với best seed
random.seed(BEST_SEED)
np.random.seed(BEST_SEED)
try:
    import torch
    torch.manual_seed(BEST_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(BEST_SEED)
        torch.cuda.manual_seed_all(BEST_SEED)
except:
    pass

model_baseline = PatchTST(
    h=horizon,
    input_size=best_params_patchtst['input_size'],
    patch_len=best_params_patchtst['patch_len'],
    stride=best_params_patchtst['stride'],
    revin=True,
    learning_rate=best_params_patchtst['learning_rate'],
    max_steps=best_params_patchtst['max_steps'],
    val_check_steps=10,
)

nf_model_baseline = NeuralForecast(models=[model_baseline], freq='D')

print(f"\n📊 Training trên TOÀN BỘ dữ liệu: {len(train_nf_full)} điểm")
print(f"   - Seed: {BEST_SEED}")
print(f"   - Parameters: {best_params_patchtst}")

nf_model_baseline.fit(df=train_nf_full, val_size=0)
forecast_baseline = nf_model_baseline.predict()

# Lấy predictions
pred_col = [col for col in forecast_baseline.columns if col not in ['unique_id', 'ds']][0]
pred_baseline = forecast_baseline[pred_col].values[:horizon]

# Đánh giá trên test ground truth
mse_baseline = mean_squared_error(test_ground_truth, pred_baseline)
bias_baseline = np.mean(pred_baseline - test_ground_truth)

print(f"\n✅ Baseline Model Results:")
print(f"   - MSE: {mse_baseline:.4f}")
print(f"   - Bias: {bias_baseline:.4f}")


## 5. Post-Processing và Bias Correction

Áp dụng post-processing regression và smooth bias correction giống như `patchtst_best_method.ipynb`:


In [ ]:
# Post-processing với TimeSeriesSplit để học mapping
print("="*70)
print("🔧 POST-PROCESSING REGRESSION")
print("="*70)

# Thu thập dữ liệu từ validation folds
tscv = TimeSeriesSplit(n_splits=3)
X_reg = []
y_reg = []

for train_idx, val_idx in tscv.split(train_nf_full):
    train_fold = train_nf_full.iloc[train_idx]
    val_fold = train_nf_full.iloc[val_idx]
    
    # Train model trên fold
    model_fold = PatchTST(
        h=len(val_fold),
        input_size=best_params_patchtst['input_size'],
        patch_len=best_params_patchtst['patch_len'],
        stride=best_params_patchtst['stride'],
        revin=True,
        learning_rate=best_params_patchtst['learning_rate'],
        max_steps=best_params_patchtst['max_steps'],
        val_check_steps=10,
    )
    
    nf_model_fold = NeuralForecast(models=[model_fold], freq='D')
    nf_model_fold.fit(df=train_fold, val_size=0)
    forecast_fold = nf_model_fold.predict()
    
    pred_fold = forecast_fold[pred_col].values
    true_fold = val_fold['y'].values
    
    X_reg.extend(pred_fold)
    y_reg.extend(true_fold)

# Train Linear Regression
X_reg = np.array(X_reg).reshape(-1, 1)
y_reg = np.array(y_reg)

reg = LinearRegression()
reg.fit(X_reg, y_reg)

print(f"\n✅ Đã học post-processing regression:")
print(f"   - Coef: {reg.coef_[0]:.4f}")
print(f"   - Intercept: {reg.intercept_:.4f}")

# Áp dụng post-processing
pred_postprocessed = reg.predict(pred_baseline.reshape(-1, 1))

# Smooth bias correction (Linear 20%)
n_smooth = int(horizon * 0.2)
weights = np.linspace(0, 1, n_smooth)
pred_final = pred_baseline.copy()
pred_final[:n_smooth] = (1 - weights) * pred_baseline[:n_smooth] + weights * pred_postprocessed[:n_smooth]
pred_final[n_smooth:] = pred_postprocessed[n_smooth:]

# Đánh giá
mse_final = mean_squared_error(test_ground_truth, pred_final)
bias_final = np.mean(pred_final - test_ground_truth)

print(f"\n✅ Final Results (với Post-processing):")
print(f"   - MSE: {mse_final:.4f} (cải thiện {((mse_baseline - mse_final) / mse_baseline * 100):.2f}%)")
print(f"   - Bias: {bias_final:.4f}")
